# 🤖 手勢石頭-剪刀-布（0, 2, 5）模型訓練與匯出 TFLite
本筆記展示如何使用 Sign Language Digits Dataset 中的部分資料（只保留 0、2、5）訓練一個輕量級分類模型，以辨識簡單手勢：石頭、剪刀、布，並將訓練完成的模型轉換為 TensorFlow Lite 格式，方便在手機端部署（例如 Chaquopy）。

## ⚙️ 步驟 1：安裝 Kaggle 並下載資料集

In [ ]:
# 安裝 Kaggle CLI 工具
!pip install kaggle --quiet

# 從 Kaggle 下載 Sign Language Digits 資料集（共 0-9 十個手勢圖片）
!kaggle datasets download -d ardamavi/sign-language-digits-dataset

# 解壓縮下載的 ZIP 檔案
!unzip -q sign-language-digits-dataset.zip


## 📁 步驟 2：過濾並載入影像（僅保留 0, 2, 5）

In [ ]:
import pathlib, cv2, numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# 我們只選取三個數字作為簡化版的石頭（0）、剪刀（2）、布（5）
KEEP_DIGITS = [0, 2, 5]
LABEL_MAP = {0: 0, 2: 1, 5: 2}  # 對應新類別編號：0=石頭, 1=剪刀, 2=布

X, y = [], []

# 遍歷目錄，讀入對應類別的影像
for digit in KEEP_DIGITS:
    for img_path in pathlib.Path(f"Dataset/{digit}").glob("*.jpg"):
        img = cv2.imread(str(img_path))                          # 使用 OpenCV 讀取圖像
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)              # 將 BGR 轉為 RGB
        img = cv2.resize(img, (96, 96)) / 255.0                 # Resize 並正規化到 0~1
        X.append(img)
        y.append(LABEL_MAP[digit])                              # 將原始數字轉換為新編號（0,1,2）

# 將資料轉為 NumPy 格式
X = np.array(X, dtype="float32")
y = to_categorical(y, num_classes=3)  # 將標籤做 one-hot 編碼（符合 softmax 輸出）


## 🧠 步驟 3：建立 MobileNetV2 模型（3 分類）

In [ ]:
from tensorflow.keras import layers, models

# 載入預訓練的 MobileNetV2（不含最上層）
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # 冷凍 backbone，不參與訓練

# 建立我們自己的分類器模型
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),           # 轉成向量
    layers.Dense(3, activation="softmax")      # 三個類別輸出（石頭、剪刀、布）
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# 分割訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y
)

# 開始訓練模型
model.fit(
    X_train, y_train,
    epochs=8,
    batch_size=32,
    validation_data=(X_test, y_test)
)


## 💾 步驟 4：轉換模型為 TensorFlow Lite

In [ ]:
# 將訓練好的模型轉換為 TFLite 格式
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# 儲存為 rps.tflite（Rock, Paper, Scissors）
with open("rps.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ rps.tflite 已成功產生！請在左側檔案總管中下載。")
